# Lexical Diversity

In this exercise, we will calculate the **type-token ratio (TTR)**, then the **moving-average type-token ratio (MATTR)**, and compare their behavior on chapters of *The Great Gatsby*.

Note: this is a word-based measure, not a lemmatized one, so *run* and *running* count as different types.

## 1. Pre-processing

Before calculating a score, decide what counts as a word. Our `tokenize` function lowercases text (`The` >  `the`) and uses a simple regular expression to keep alphabetic words (no punctuation and numbers) and internal apostrophes (`Nick's`). The preprocessing rule matters: changing it can change both the token count and the type count.

Run the next cell. Count the tokens and unique types in the sample, then calculate the TTR by hand: types divided by tokens.

In [6]:
import re
from pathlib import Path


def tokenize(text):
    # Brief description of the function
    """Return lowercase word tokens, keeping internal apostrophes."""

    # Finds all matches of one or more letters
    # [A-Za-z]+
    # optionally followed by an apostrophe (' or curly ’) and more letters
    # (?:['’][A-Za-z]+)?
    # 
    # Normalize case so capitalization does not create extra word types.
    # 
    # Return every match in a list of strings
    return re.findall(r"[A-Za-z]+(?:['’][A-Za-z]+)?", text.lower())

In [7]:
sample_text = "Nick saw Gatsby's green light; Gatsby saw Nick."
sample_tokens = tokenize(sample_text)
# A set keeps each distinct token once, so its length is the type count.
sample_types = set(sample_tokens)

print("Tokens:", sample_tokens)
print("Token count:", len(sample_tokens))
print("Types:", sorted(sample_types))
print("Type count:", len(sample_types))

Tokens: ['nick', 'saw', "gatsby's", 'green', 'light', 'gatsby', 'saw', 'nick']
Token count: 8
Types: ['gatsby', "gatsby's", 'green', 'light', 'nick', 'saw']
Type count: 6


## 2. Calculate TTR

The simple type-token ratio is

$$\mathrm{TTR} = \frac{\text{number of types}}{\text{number of tokens}}.$$

For a non-empty text, TTR is greater than 0 and at most 1. A higher value means a larger share of the tokens are different types in this sample. 

Note: This does not mean that the writer knows more words, because as we said TTR is sensitive to text length and to the text's topic and content.

Calculate manually both scores before running the next cell.

In [8]:
def ttr(tokens):
    """Calculate types / tokens for a non-empty sequence of tokens."""
    if not tokens:
        raise ValueError("TTR is undefined for an empty token sequence.")
    # Count each word form once, then divide by all running word tokens.
    return len(set(tokens)) / len(tokens)


short_tokens = tokenize("red blue green")
longer_tokens = tokenize("red blue green red blue green red blue")

print(f"Short text:  {len(set(short_tokens))} types / {len(short_tokens)} tokens = {ttr(short_tokens):.3f}")
print(f"Longer text: {len(set(longer_tokens))} types / {len(longer_tokens)} tokens = {ttr(longer_tokens):.3f}")

Short text:  3 types / 3 tokens = 1.000
Longer text: 3 types / 8 tokens = 0.375


## 3. Calculate MATTR

For a token sequence of length $n$ and window size $w$, move the window forward **one token at a time**. That makes the windows overlap. There are $n-w+1$ windows, and MATTR is the mean of their TTR values. A window size must be a positive integer and no larger than the text.

Before running the next cell, predict how many windows an 8-token text will have with a window size of 4.

In [9]:
def mattr(tokens, window_size=50):
    """Return the mean TTR of all one-token-shifted windows."""
    if not isinstance(window_size, int) or window_size <= 0:
        raise ValueError("window_size must be a positive integer.")
    if window_size > len(tokens):
        raise ValueError("window_size cannot exceed the number of tokens.")

    # Shift the window by one token to include every overlapping segment.
    window_ttrs = [
        ttr(tokens[start:start + window_size])
        for start in range(len(tokens) - window_size + 1)
    ]
    return sum(window_ttrs) / len(window_ttrs)


window_demo = tokenize("red blue red green blue red yellow green")
window_size = 4
# Show each window's TTR so the averaging step is visible.
window_scores = [
    ttr(window_demo[start:start + window_size])
    for start in range(len(window_demo) - window_size + 1)
]

print("Number of overlapping windows:", len(window_scores))
print("TTR for each window:", [round(score, 3) for score in window_scores])
print(f"MATTR (window size {window_size}): {mattr(window_demo, window_size):.3f}")

Number of overlapping windows: 5
TTR for each window: [0.75, 0.75, 0.75, 1.0, 1.0]
MATTR (window size 4): 0.850


## 4. Compare *The Great Gatsby* chapters

The default folder is `../data/FITZ-TGG-25_chapters/`. In the next cell, change `corpus_dir` to the folder containing another corpus of `.txt` files. The cell analyzes every `.txt` file directly inside that folder, using 50-token windows for MATTR.

For the Gatsby chapters, run the cell to compare token count, type count, TTR, and MATTR. Then answer the questions below:

1. Which chapter has the highest TTR? Which has the highest MATTR?
2. Do the rankings match? Suggest one reason they might differ.
3. What changes when you use a 100-token window instead of 50? Compare the scores, not just their order.
4. What does this word-based exercise miss when it treats *walk*, *walks*, and *walking* as different types?

In [11]:
# Change this folder to analyze a different corpus of .txt files.
corpus_dir = Path("../data/FITZ-TGG-25_chapters")  # Change the path to "../data" to run on our corpus
text_files = sorted(corpus_dir.glob("*.txt"))
window_size = 50
text_results = []

for text_file in text_files:
    # Apply the same tokenization rule and MATTR window to every text.
    text_tokens = tokenize(text_file.read_text(encoding="utf-8"))
    text_results.append(
        {
            "file": text_file.stem,
            "tokens": len(text_tokens),
            "types": len(set(text_tokens)),
            "ttr": ttr(text_tokens),
            "mattr": mattr(text_tokens, window_size),
        }
    )

# Print the results in a formatted table.
# <10 means left-align in a 10-character-wide column; >10 means right-align in a 10-character-wide column
print(f"{'File':<10}{'Tokens':>10}{'Types':>10}{'TTR':>10}{f'MATTR ({window_size})':>16}")
for result in text_results:
    print(
        f"{result['file']:<10}{result['tokens']:>10}{result['types']:>10}"
        f"{result['ttr']:>10.3f}{result['mattr']:>16.3f}"
    )

File          Tokens     Types       TTR      MATTR (50)
CATH-ALL-23     34256      4510     0.132           0.830
FITZ-TGG-25     48684      5940     0.122           0.826
HEMI-TSAR-26     68542      4859     0.071           0.767


# Bonus exercise

Explore the API of the Folger Shakespeare: https://www.folgerdigitaltexts.org/api.

Calculate TTR and MATTR for the monologues in each play.